In [2]:
from fastai.collab import *
from fastai.tabular.all import *

In [3]:
path = untar_data(URLs.ML_100k)

In [4]:
ratings = pd.read_csv(path/'u.data', delimiter='\t', header=None,      #they are tab seperated values
                      names=['user','movie','rating','timestamp'])
movies = pd.read_csv(path/'u.item',  delimiter='|', encoding='latin-1',
                     usecols=(0,1), names=('movie','title'), header=None)  # this has movie no followed with name so we will merge both
ratings=ratings.merge(movies)


In [5]:
dls = CollabDataLoaders.from_df(ratings, item_name='title', bs=64)
dls.show_batch()

,user,title,rating
0,185,Mother (1996),5
1,456,Contact (1997),4
2,940,Air Force One (1997),5
3,413,"Game, The (1997)",2
4,326,"Blob, The (1958)",4
5,862,"Addiction, The (1995)",4
6,374,"Stupids, The (1996)",1
7,548,Strange Days (1995),5
8,6,Ponette (1996),4
9,190,Absolute Power (1997),3


In [6]:
learn = collab_learner(dls, n_factors=50, y_range=(0, 5.5))

In [7]:
learn.fit_one_cycle(5, 5e-3, wd=0.1)

epoch,train_loss,valid_loss,time
0,0.912944,0.953415,00:12
1,0.670251,0.899107,00:17
2,0.527844,0.876112,00:20
3,0.477687,0.861537,00:18
4,0.447019,0.858033,00:19


In [8]:
movie_bias = learn.model.i_bias.weight.squeeze()
idxs = movie_bias.argsort(descending=True)[:5]
[dls.classes['title'][i] for i in idxs]

['Titanic (1997)',
 'Star Wars (1977)',
 'Shawshank Redemption, The (1994)',
 'Good Will Hunting (1997)',
 "Schindler's List (1993)"]

In [9]:
# %%
# We only need the movie-side parameters + the movie list to serve new users later.
import numpy as np
import pickle

n_users, n_factors_used = learn.model.u_weight.weight.shape
rating_counts = ratings.groupby('title').size()
popular_titles = rating_counts.sort_values(ascending=False).head(20).index.tolist()

movie_factors = learn.model.i_weight.weight.data.cpu().numpy()   # (n_movies, n_factors)
movie_bias    = learn.model.i_bias.weight.data.cpu().numpy().squeeze()  # (n_movies,)
movie_titles  = list(dls.classes['title'])

# y_range used when training (so we replicate the same squashing at inference)
y_range = (0, 5.5)

with open('movie_model.pkl', 'wb') as f:
    pickle.dump({
        'movie_factors': movie_factors,
        'movie_bias': movie_bias,
        'movie_titles': movie_titles,
        'y_range': y_range,
        'popular_titles': popular_titles,
    }, f)

In [10]:
# %%
import torch
import torch.nn.functional as F

def fit_new_user_embedding(rated_indices, ratings_given, movie_factors, movie_bias,
                            y_range=(0, 5.5), n_epochs=200, lr=0.05, wd=0.01):
    """
    rated_indices: list[int] - indices into movie_titles/movie_factors the user rated
    ratings_given: list[float] - the ratings (0-5) the user gave, same order
    Returns: (user_vec, user_bias) fitted for this one new user.
    """
    n_factors = movie_factors.shape[1]
    movie_factors_t = torch.tensor(movie_factors, dtype=torch.float32)
    movie_bias_t = torch.tensor(movie_bias, dtype=torch.float32)

    idx = torch.tensor(rated_indices, dtype=torch.long)
    y = torch.tensor(ratings_given, dtype=torch.float32)

    # Initialize new user embedding near zero (small random), plus a scalar bias
    user_vec = torch.zeros(n_factors, requires_grad=True)
    user_bias = torch.zeros(1, requires_grad=True)

    opt = torch.optim.Adam([user_vec, user_bias], lr=lr, weight_decay=wd)

    lo, hi = y_range
    for epoch in range(n_epochs):
        opt.zero_grad()
        rated_movie_factors = movie_factors_t[idx]        # (k, n_factors)
        rated_movie_bias = movie_bias_t[idx]               # (k,)

        dot = (rated_movie_factors * user_vec).sum(dim=1)  # (k,)
        raw_pred = dot + rated_movie_bias + user_bias
        pred = torch.sigmoid(raw_pred) * (hi - lo) + lo     # same sigmoid_range trick fastai uses

        loss = F.mse_loss(pred, y)
        loss.backward()
        opt.step()

    return user_vec.detach().numpy(), user_bias.detach().item()

In [11]:
# %%
def recommend_for_new_user(rated_titles_and_scores, movie_titles, movie_factors, movie_bias,
                            y_range=(0, 5.5), top_n=10):
    """
    rated_titles_and_scores: dict like {"Star Wars (1977)": 5, "Titanic (1997)": 2}
    """
    title_to_idx = {t: i for i, t in enumerate(movie_titles)}
    rated_indices = [title_to_idx[t] for t in rated_titles_and_scores if t in title_to_idx]
    ratings_given = [rated_titles_and_scores[movie_titles[i]] for i in rated_indices]

    user_vec, user_bias = fit_new_user_embedding(
        rated_indices, ratings_given, movie_factors, movie_bias, y_range
    )

    movie_factors_t = torch.tensor(movie_factors, dtype=torch.float32)
    movie_bias_t = torch.tensor(movie_bias, dtype=torch.float32)
    user_vec_t = torch.tensor(user_vec, dtype=torch.float32)

    raw_pred = (movie_factors_t * user_vec_t).sum(dim=1) + movie_bias_t + user_bias
    lo, hi = y_range
    preds = torch.sigmoid(raw_pred) * (hi - lo) + lo

    # Mask out movies already rated
    preds_np = preds.detach().numpy()
    preds_np[rated_indices] = -np.inf

    top_idx = np.argsort(-preds_np)[:top_n]
    return [(movie_titles[i], float(preds_np[i])) for i in top_idx]